# A support bot in your brand's voice: MiniCPM5 1B, SFT then DPO, on a Mac

**Kettlebird** is a made-up online shop for kitchen and home goods, and **Wren** is its support assistant.
We teach OpenBMB's **MiniCPM5 1B** to answer customers the way Wren should: short, answer first, plain words, signed.
Two kinds of training with **mlx-tune** on the Mac's GPU: **SFT** (copy good answers), then **DPO** (prefer the
on-brand answer over the generic one).

In [ ]:
import json
import time

import mlx.core as mx
from mlx.utils import tree_flatten
from mlx_lm import load

from kettlebird import (BASE, TEACHER, STYLE_GUIDE, answer_all, check, fill, free, judge, load_bitext, load_pairs,
                        prompt_for, rewrite, rule_report, scoreboard, test_questions)

## 0. The machine

The same MacBook Pro as every lesson in this series. MLX runs on the Mac's GPU.

In [ ]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

## 1. The brand

What "on brand" means for Kettlebird, written down as a short style guide. Rules 1, 2, 4, 7 and 8 can be checked by
code; how plain and warm a reply sounds needs a reader, so a judge model checks that later.

In [ ]:
print(STYLE_GUIDE)

## 2. The data: Bitext customer support

[`bitext/Bitext-customer-support-llm-chatbot-training-dataset`](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset)
(CDLA-Sharing-1.0): 26,872 customer messages over 27 intents, each with a generic support answer.
The messages are what we want; the answers are the voice we **don't** want.

In [ ]:
d = load_bitext()
print(len(d), "rows,", d.intent.nunique(), "intents")
print(d.instruction[0])
print(d.response[0])

In [ ]:
cs = [check(a) for a in d.response]
print(f"Bitext's answers: sign-off {sum(c['signoff'] for c in cs)} / {len(cs)},  "
      f"80 words or fewer {100 * sum(c['short'] for c in cs) / len(cs):.0f} %,  "
      f"mean {sum(c['words'] for c in cs) / len(cs):.0f} words")

Bitext writes `{{Order Number}}` where a real order number would be. We fill every placeholder with a simple
made-up value (an order number like KB-34391, Kettlebird's phone and links), so answers never print `{{...}}`.

The test set: 4 messages per intent = **108**, never used for training.

In [ ]:
test = [fill(t) for t in test_questions(d)]
questions = [t["question"] for t in test]
hook = questions[5]
len(test), hook

## 3. Before: the base model

MiniCPM5 1B, OpenBMB's official 4-bit MLX build. Every model in this notebook gets the same short system prompt:
*You are Wren, the customer support assistant for Kettlebird, an online shop for kitchen and home goods.*

In [ ]:
model, tok = load(BASE)
print(answer_all(model, tok, [hook])[0])

In [ ]:
base = answer_all(model, tok, questions)
rule_report(base)

### "Just put the style guide in the prompt"

The obvious first try: give the base model the whole style guide in its system prompt.

In [ ]:
guided = answer_all(model, tok, questions, guide=True)
rule_report(guided)
print(guided[5])

In [ ]:
del model
free()

## 4. On-brand answers to learn from

A bigger local model, **Qwen 3.8 27B** (4-bit MLX), rewrites Bitext's generic answers in the Kettlebird voice:
same facts and steps, new voice. Here it rewrites three. (It is a 16 GB download and needs about 26 GB of memory;
skip these two cells on a smaller Mac: the ready file below already has the rewrites.)

In [ ]:
pairs = [fill(p) for p in load_pairs()]
teacher, teacher_tok = load(TEACHER)
new = rewrite(teacher, teacher_tok, [{"question": p["question"], "bitext": p["rejected"]} for p in pairs[:3]])
for p, n in zip(pairs, new):
    print("CUSTOMER:", p["question"])
    print("BITEXT  :", p["rejected"][:300], "...")
    print("QWEN 27B:", n, "\n")

In [ ]:
del teacher
free()

I made 810 of these this way (30 per intent, about 33 minutes on this Mac). They are in `kettlebird_pairs.jsonl`:
the customer message, Bitext's answer (**rejected**) and the rewrite (**chosen**). We keep the rewrites that pass
every rule check: the first 540 teach SFT, the rest are DPO's pairs.

In [ ]:
ok = [p for p in pairs if check(p["chosen"])["all"]]
sft_rows, dpo_rows = ok[:540], ok[540:]
len(pairs), len(ok), len(sft_rows), len(dpo_rows)

## 5. SFT: learn the voice from good answers

LoRA (rank 16) on all seven linear layers of the 24 blocks; the 4-bit base stays frozen. Each sample is the exact
text the model sees when it answers (system prompt, message, the empty `<think></think>` of non-thinking mode) plus
the on-brand answer.

In [ ]:
from mlx_tune import FastLanguageModel, SFTConfig, SFTTrainer

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=1024)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32)
sft_data = [{"text": prompt_for(tok, p["question"]).removeprefix("<s>") + p["chosen"] + "<|im_end|>"} for p in sft_rows]
print(sft_data[0]["text"])

In [ ]:
args = SFTConfig(output_dir="runs/sft", per_device_train_batch_size=4, learning_rate=1e-4, max_steps=150,
                 warmup_steps=10, logging_steps=10, save_steps=50, max_seq_length=1024, val_batches=0)
mx.reset_peak_memory()
t0 = time.time()
SFTTrainer(model=model, tokenizer=tok, train_dataset=sft_data, args=args).train()
sft_secs, sft_peak = time.time() - t0, mx.get_peak_memory() / 1e9
trainable = sum(v.size for _, v in tree_flatten(model.model.trainable_parameters()))
print(f"SFT: {sft_secs:.1f} s, peak memory {sft_peak:.2f} GB, trainable LoRA parameters {trainable:,}")

In [ ]:
del model
free()
model, tok = load(BASE, adapter_path="runs/sft/adapters")
sft = answer_all(model, tok, questions)
rule_report(sft)
print(sft[5])

In [ ]:
del model
free()

## 6. DPO: prefer the on-brand answer

DPO sees the same message twice: the **chosen** answer and the **rejected** one, and pushes the model towards the
first. It starts from the SFT model, which is also its frozen reference.
(Two lines after loading the adapter: mlx-lm leaves the norm weights trainable, so we train only the LoRA matrices.
Without them, the saved DPO adapter would not load again.)

In [ ]:
from mlx_tune import DPOConfig, DPOTrainer

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=1024)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32)
model.load_adapter("runs/sft/adapters")
model.model.freeze()
model.model.unfreeze(keys=["lora_a", "lora_b"], strict=False)
model.model.train()
dpo_data = [{"prompt": prompt_for(tok, p["question"]).removeprefix("<s>"),
             "chosen": p["chosen"] + "<|im_end|>", "rejected": p["rejected"] + "<|im_end|>"} for p in dpo_rows]
print("CHOSEN  :", dpo_data[0]["chosen"])
print("REJECTED:", dpo_data[0]["rejected"])

In [ ]:
args = DPOConfig(output_dir="runs/dpo", beta=0.1, learning_rate=5e-6, per_device_train_batch_size=2,
                 max_steps=100, logging_steps=10, save_steps=50, max_seq_length=1024)
mx.reset_peak_memory()
t0 = time.time()
DPOTrainer(model=model, tokenizer=tok, train_dataset=dpo_data, args=args).train()
dpo_secs, dpo_peak = time.time() - t0, mx.get_peak_memory() / 1e9
print(f"DPO: {dpo_secs:.1f} s, peak memory {dpo_peak:.2f} GB")

In [ ]:
del model
free()
model, tok = load(BASE, adapter_path="runs/dpo/adapters")
dpo = answer_all(model, tok, questions)
rule_report(dpo)
print(dpo[5])

In [ ]:
del model
free()

## 7. The judge and the scoreboard

The code checks the rules. For the voice, Qwen 3.8 27B reads the style guide, the message and one answer at a time,
without knowing which model wrote it, and scores it 1 to 5. (It is the same model that wrote the rewrites, so it
may like its own style.) **On brand** = every rule check passes **and** the judge gives 4 or 5.

In [ ]:
answers = {"base": base, "SFT": sft, "SFT + DPO": dpo}
judge_model, judge_tok = load(TEACHER)
scores = judge(judge_model, judge_tok, answers, questions)
del judge_model
free()

In [ ]:
on_brand = scoreboard(answers, scores)
json.dump({"questions": questions, "answers": answers, "scores": scores}, open("runs/results.json", "w"), indent=1)

In [ ]:
for i in [5, 80, 26]:
    print("CUSTOMER :", questions[i])
    print("BASE     :", base[i], "\n")
    print("SFT + DPO:", dpo[i], "\n")

In [ ]:
print(f"MacBook Pro M5 Max, 36 GB | MiniCPM5 1B, LoRA | SFT {sft_secs:.0f} s + DPO {dpo_secs:.0f} s | "
      f"peak {max(sft_peak, dpo_peak):.1f} GB")
print(f"on brand: base {on_brand['base']:.1f} %  ->  SFT {on_brand['SFT']:.1f} %  ->  SFT + DPO {on_brand['SFT + DPO']:.1f} %")

## 8. Run it in Ollama

Merge the LoRA into the model (`mlx_lm fuse`, back to 16-bit), turn it into a GGUF file with llama.cpp's converter and
shrink it to Q4_K_M. We use the llama.cpp that Unsloth Studio installed in lesson 4 (its Python has the converter's
packages). MiniCPM5 is a thinking model, so we run it with `--think=false`, the mode it was trained in.

In [ ]:
import sys
from pathlib import Path

from kettlebird import BASE

LLAMA_CPP = Path.home() / ".unsloth/llama.cpp"
STUDIO_PYTHON = Path.home() / ".unsloth/studio/unsloth_studio/bin/python"
!{sys.executable} -m mlx_lm fuse --model {BASE} --adapter-path runs/dpo/adapters --save-path runs/kettlebird-fused --dequantize
!{STUDIO_PYTHON} {LLAMA_CPP}/convert_hf_to_gguf.py runs/kettlebird-fused --outfile runs/kettlebird-f16.gguf --outtype f16 > runs/convert.log 2>&1; tail -1 runs/convert.log
!{LLAMA_CPP}/llama-quantize runs/kettlebird-f16.gguf runs/kettlebird-Q4_K_M.gguf Q4_K_M > runs/quantize.log 2>&1; tail -2 runs/quantize.log
!ls -lh runs/*.gguf

In [ ]:
from kettlebird import SYSTEM

open("runs/Modelfile", "w").write(f'''FROM ./kettlebird-Q4_K_M.gguf
TEMPLATE """<|im_start|>system
{{{{ .System }}}}<|im_end|>
<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
<think>

</think>

"""
SYSTEM """{SYSTEM}"""
PARAMETER stop <|im_end|>
PARAMETER temperature 0
''')
!cd runs && ollama create kettlebird -f Modelfile

## 9. Put it on Hugging Face

The LoRA adapter (for MLX) and the Q4_K_M file (for Ollama) go to a Hugging Face model repo.
Log in once with `hf auth login`.

In [ ]:
from huggingface_hub import HfApi

REPO = "kgptalkie/minicpm5-1b-kettlebird-wren"
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path="runs/dpo/adapters", repo_id=REPO, allow_patterns=["adapters.safetensors", "adapter_config.json"])
api.upload_file(path_or_fileobj="runs/kettlebird-Q4_K_M.gguf", path_in_repo="kettlebird-Q4_K_M.gguf", repo_id=REPO)
api.upload_file(path_or_fileobj="MODEL_CARD.md", path_in_repo="README.md", repo_id=REPO)
print(f"https://huggingface.co/{REPO}")